Ingest raw files to databricks

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_inv;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_inv.bronze;
CREATE SCHEMA IF NOT EXISTS retail_inv.silver;
CREATE SCHEMA IF NOT EXISTS retail_inv.gold;
CREATE SCHEMA IF NOT EXISTS retail_inv.ml;
CREATE VOLUME IF NOT EXISTS retail_inv.bronze.raw_files;

In [0]:
from pyspark.sql import functions as F

base = "/Volumes/retail_inv/bronze/raw_files/m5/"

def ingest(file_name, table_name):
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(base + file_name)
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
    df.write.format("delta").mode("overwrite").saveAsTable(f"retail_inv.bronze.{table_name}")
    print(table_name, spark.table(f"retail_inv.bronze.{table_name}").count())

ingest("calendar.csv", "calendar")
ingest("sell_prices.csv", "sell_prices")
ingest("sales_train_validation.csv", "sales_wide")

calendar 1969
sell_prices 6841121
sales_wide 30490
